# SafeDecoding — chạy thử trên Google Colab (GPU free)

Mục đích: **chứng minh method chạy được thật + lấy số DSR** trên sample nhỏ (máy Mac không có GPU CUDA nên chạy ở đây).

**Bước bắt buộc trước khi chạy:** menu `Runtime` → `Change runtime type` → chọn **T4 GPU** → `Save`.
Sau đó `Runtime` → `Run all` (hoặc chạy từng cell theo thứ tự).


In [ ]:
# 1) Kiểm tra đã bật GPU chưa (phải thấy bảng GPU Tesla T4). Nếu lỗi -> chưa bật GPU runtime.
!nvidia-smi


In [ ]:
# 2) Tải repo + cài ĐÚNG version đã verify (KHÔNG đụng torch của Colab vì nó sẵn CUDA)
!git clone -q https://github.com/uw-nsl/SafeDecoding.git
%cd /content/SafeDecoding
!pip install -q "numpy==1.26.4" "pyarrow==12.0.1" "huggingface_hub==0.16.4" "datasets==2.14.6" \
  "transformers==4.28.1" "tokenizers==0.13.3" "accelerate==0.25.0" "fschat==0.2.20" \
  "sentencepiece" "ml_collections" "tenacity==8.2.3"
!pip install -q -e ./peft
print('>> Cài xong. Nếu Colab hiện nút "RESTART RUNTIME" (do hạ numpy) -> bấm Restart, rồi chạy lại TỪ cell này.')


In [ ]:
# 3) Tạo sample 10 prompt (id/goal/prompt) lấy từ AdvBench có sẵn trong repo
import json
adv = json.load(open('/content/SafeDecoding/datasets/advbench_harmful_behaviors.json'))
idx = [round(i*(len(adv)-1)/9) for i in range(10)]
sample = [{'id': k, 'goal': adv[j]['goal'], 'prompt': adv[j]['goal']} for k, j in enumerate(idx)]
json.dump(sample, open('/content/SafeDecoding/datasets/custom_prompts.json','w'), indent=4)
print('Đã ghi', len(sample), 'prompt sample vào datasets/custom_prompts.json')


In [ ]:
# 4) CHẠY SafeDecoding (CÓ phòng thủ) - chấm offline DictJudge, KHÔNG cần API key
#    Lần đầu sẽ tải vicuna-7b (~13GB) -> đợi vài phút. Cuối log có dòng 'ASR: xx.xx%'.
%cd /content/SafeDecoding/exp
!python defense.py --model_name vicuna --attacker custom --defender SafeDecoding --disable_GPT_judge --max_new_tokens 256


In [ ]:
# 5) (đối chứng) CHẠY khi TẮT phòng thủ -> DSR sẽ THẤP hơn nhiều so với cell 4
!python defense.py --model_name vicuna --attacker custom --defense_off --disable_GPT_judge --max_new_tokens 256


## Đọc kết quả
- Cuối mỗi lần chạy có dòng `ASR: xx.xx%` -> **DSR (độ phòng thủ) = 100 − ASR**.
- **Có** SafeDecoding (cell 4): DSR **cao** (model từ chối nhiều). **Tắt** phòng thủ (cell 5): DSR **thấp**. Chênh lệch đó = bằng chứng method có tác dụng.
- File chi tiết (câu trả lời + thời gian) lưu ở `/content/SafeDecoding/exp/exp_outputs/...`.
- ⚠️ Nếu **OOM** (hết VRAM T4): giảm `--max_new_tokens 128`, hoặc sửa số `10` ở cell 3 xuống `5`.
- Muốn dùng đúng repo của mình (đã pin sẵn + có fix): đổi link `git clone` ở cell 2 sang repo GitHub anh đã push.
